In [1]:
import warnings
import numpy as np # for mathematical operrations
import pandas as pd
import plotly.express as px  # for visualisatons
import plotly. graph_objects as go  # for one of the visualisations
from sklearn.impute import SimpleImputer # for imputing missing values
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline,make_pipeline # combine things from sklearn



In [2]:
def wrangle(filepath):
    # read csv file into DataFrame
    df= pd.read_csv(filepath)

    #subset to properties that are in Capital federal
    mask_ba = df['place_with_parent_names'].str.contains('Capital Federal')
    
    #subset to properties that are apartments
    mask_apt= df['property_type'] == 'apartment'

    # subset to properties less than $400,000 USD
    mask_price= df['price_aprox_usd'] < 400_000

    

    df = df[mask_ba & mask_apt & mask_price]

    # subset outliers by `surface_covered_in_m2`
    low,high= df['surface_covered_in_m2'].quantile([0.1,0.9])
    mask_area= df['surface_covered_in_m2'].between(low,high)
    df= df[mask_area]

    #split the 'lat-lon' column
    df[['lat','lon']]= df['lat-lon'].str.split(',', expand=True).astype(float)
    df.drop(columns='lat-lon',inplace=True)
    
    return df



In [3]:
frame1= wrangle(r"C:\Users\Peace\OneDrive\WP projects\HOUSING IN BUENOS AIRES 2\Bueinos-Aires-Real_estate-1 - Sheet1.csv")
print(frame1.info())
frame1.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1343 entries, 4 to 8604
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   1343 non-null   object 
 1   property_type               1343 non-null   object 
 2   place_with_parent_names     1343 non-null   object 
 3   price                       1343 non-null   float64
 4   currency                    1343 non-null   object 
 5   price_aprox_local_currency  1343 non-null   float64
 6   price_aprox_usd             1343 non-null   float64
 7   surface_total_in_m2         965 non-null    float64
 8   surface_covered_in_m2       1343 non-null   float64
 9   price_usd_per_m2            927 non-null    float64
 10  price_per_m2                1343 non-null   float64
 11  floor                       379 non-null    float64
 12  rooms                       1078 non-null   float64
 13  expenses                    349 non-nu

,operation,property_type,place_with_parent_names,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,index,lat,lon
4,sell,apartment,|Argentina|Capital Federal|Chacarita|,129000.0,USD,1955949.6,129000.0,76.0,70.0,1697.368421,1842.857143,NaN,NaN,NaN,http://chacarita.properati.com.ar/10qlv_venta_...,5,-34.584651,-58.454693
9,sell,apartment,|Argentina|Capital Federal|Villa Luro|,87000.0,USD,1319128.8,87000.0,48.0,42.0,1812.500000,2071.428571,NaN,NaN,NaN,http://villa-luro.properati.com.ar/12m82_venta...,10,-34.638979,-58.500115
29,sell,apartment,|Argentina|Capital Federal|Caballito|,118000.0,USD,1789163.2,118000.0,NaN,54.0,NaN,2185.185185,NaN,2.0,NaN,http://caballito.properati.com.ar/11wqh_venta_...,30,-34.615847,-58.459957
40,sell,apartment,|Argentina|Capital Federal|Constitución|,57000.0,USD,864256.8,57000.0,42.0,42.0,1357.142857,1357.142857,5.0,2.0,364,http://constitucion.properati.com.ar/k2f0_vent...,41,-34.625222,-58.382382
41,sell,apartment,|Argentina|Capital Federal|Once|,90000.0,USD,1364616.0,90000.0,57.0,50.0,1578.947368,1800.000000,NaN,3.0,450,http://once.properati.com.ar/suwa_venta_depart...,42,-34.610610,-58.412511


for this model we consider the apartment location. however the latitude and longitude data is joined in one column and is of object rather than float.

In [4]:
frame1[['lat','lon']]= frame1['lat-lon'].str.split(',',expand=True).astype(float)
frame1.drop(columns= 'lat-lon',inplace=True)
frame1.info()

KeyError: 'lat-lon'

now that thee wrangle function works, we use it to clean and import more data.

In [ ]:
frame2=wrangle(r"C:\Users\Peace\OneDrive\WP projects\HOUSING IN BUENOS AIRES 2\bueinos- aires-real-estate-2 - Sheet1.csv")


we now need to combine(concatenate) the dataframes

In [ ]:
df= pd.concat([frame1,frame2], ignore_index=True)
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2658 entries, 0 to 2657
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   2658 non-null   object 
 1   property_type               2658 non-null   object 
 2   place_with_parent_names     2658 non-null   object 
 3   price                       2658 non-null   float64
 4   currency                    2658 non-null   object 
 5   price_aprox_local_currency  2658 non-null   float64
 6   price_aprox_usd             2658 non-null   float64
 7   surface_total_in_m2         1898 non-null   float64
 8   surface_covered_in_m2       2658 non-null   float64
 9   price_usd_per_m2            1818 non-null   float64
 10  price_per_m2                2658 non-null   float64
 11  floor                       769 non-null    float64
 12  rooms                       2137 non-null   float64
 13  expenses                    688 n

,operation,property_type,place_with_parent_names,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url,index,lat,lon,Index
0,sell,apartment,|Argentina|Capital Federal|Chacarita|,129000.0,USD,1955949.6,129000.0,76.0,70.0,1697.368421,1842.857143,NaN,NaN,NaN,http://chacarita.properati.com.ar/10qlv_venta_...,5.0,-34.584651,-58.454693,NaN
1,sell,apartment,|Argentina|Capital Federal|Villa Luro|,87000.0,USD,1319128.8,87000.0,48.0,42.0,1812.500000,2071.428571,NaN,NaN,NaN,http://villa-luro.properati.com.ar/12m82_venta...,10.0,-34.638979,-58.500115,NaN
2,sell,apartment,|Argentina|Capital Federal|Caballito|,118000.0,USD,1789163.2,118000.0,NaN,54.0,NaN,2185.185185,NaN,2.0,NaN,http://caballito.properati.com.ar/11wqh_venta_...,30.0,-34.615847,-58.459957,NaN
3,sell,apartment,|Argentina|Capital Federal|Constitución|,57000.0,USD,864256.8,57000.0,42.0,42.0,1357.142857,1357.142857,5.0,2.0,364,http://constitucion.properati.com.ar/k2f0_vent...,41.0,-34.625222,-58.382382,NaN
4,sell,apartment,|Argentina|Capital Federal|Once|,90000.0,USD,1364616.0,90000.0,57.0,50.0,1578.947368,1800.000000,NaN,3.0,450,http://once.properati.com.ar/suwa_venta_depart...,42.0,-34.610610,-58.412511,NaN


## Explore 

we need to show the longitude, latitude and the price in a 2 dimensional screen